In [ ]:
# correlation analysis is your next critical fairness checkpoint, because even if group representation 
# looks balanced and statistical tests look fine, bias can still sneak in through correlated features — 
# what’s called proxy bias.

# To detect hidden relationships (direct or indirect) between sensitive attributes
# (Gender, Race, Age) and other features
# (Education, Experience_Years, Skills, Screening_Score, etc.)
# that could cause unintended discrimination during selection or model training.

# In short: correlation analysis finds where bias hides behind other variables.

# Even if your model doesn’t use “Gender” explicitly, a correlated variable like Experience_Years, Education, or Certifications could still encode it.
# That’s how proxy bias forms — the model “learns” demographic patterns indirectly.

# Example: If Gender correlates strongly with Screening_Score, the selection model could favor one gender even if the “gender” column isn’t used.

# If correlation > 0.3 (or < -0.3), that’s a moderate link. Then there is potential for proxy bias.

In [2]:
# sensitive attributes = ["Gender", "Race", "Age_Group"]
# other features = ["Education", "Experience_Years", "Skills", "Screening_Score", "Job_Role_Applied", "Certifications"]
# Target variable = "Selected"

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Load dataset
df = pd.read_csv("../../datasets/recruitment.csv")
df.drop(columns=["Candidate_ID"], inplace=True)
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 65], labels=['18-25', '26-35', '36-45', '46-55', '56-65'])
df['Selected'] = df['Selected'].map({'Yes': 1, 'No': 0})
df.drop(columns=["Age"], inplace=True)

# Multi-label encoding for Skills
mlb = MultiLabelBinarizer()
skills_df = df.copy()

skills_df['Skills'] = skills_df['Skills'].fillna('').str.lower().str.split(',')

skills_encoded = pd.DataFrame(mlb.fit_transform(skills_df['Skills']),
                              columns=[f"skill_{s.replace(' ', '_')}" for s in mlb.classes_])

skills_df = pd.concat([df.drop(columns=['Skills']), skills_encoded], axis=1)

# Encode categorical variables
df_encoded = skills_df.copy()

categorical_cols = ["Gender", "Race", "Age_Group", "Education", "Job_Role_Applied", "Certifications"]
for col in categorical_cols:
    df_encoded[col] = pd.factorize(df_encoded[col])[0]

skills_cols = [col for col in df_encoded.columns if col.startswith('skill_')]
numerical_cols = ["Experience_Years", "Screening_Score", "Selected"]

df_encoded = df_encoded[categorical_cols + numerical_cols + skills_cols]

# Compute correlation matrix
corr_matrix = df_encoded.corr(method='pearson')
print(corr_matrix)

corr_matrix.style.background_gradient(cmap='coolwarm', axis=None)
# Focus on correlations between sensitive attributes and other features

                              Gender      Race  Age_Group  Education  \
Gender                      1.000000  0.013598  -0.024370  -0.011902   
Race                        0.013598  1.000000  -0.026104  -0.008531   
Age_Group                  -0.024370 -0.026104   1.000000   0.000078   
Education                  -0.011902 -0.008531   0.000078   1.000000   
Job_Role_Applied           -0.013479 -0.013117  -0.036980  -0.003336   
Certifications             -0.000863 -0.021636  -0.006754   0.010159   
Experience_Years           -0.031015 -0.035884   0.837926  -0.006599   
Screening_Score            -0.139215 -0.031625   0.008416   0.014228   
Selected                   -0.124238 -0.041165   0.019073   0.045803   
skill__agile                0.000231 -0.007720  -0.024058  -0.030620   
skill__aws                  0.016513  0.001057   0.006676   0.025195   
skill__budgeting            0.009984 -0.001815   0.013538  -0.028369   
skill__c++                  0.011234 -0.013844   0.004387   0.01

,Gender,Race,Age_Group,Education,Job_Role_Applied,Certifications,Experience_Years,Screening_Score,Selected,skill__agile,skill__aws,skill__budgeting,skill__c++,skill__cold_calling,skill__communication,skill__conflict_resolution,skill__crm,skill__deep_learning,skill__git,skill__hr_policies,skill__java,skill__leadership,skill__machine_learning,skill__negotiation,skill__networking,skill__presentation,skill__python,skill__r,skill__recruitment,skill__scheduling,skill__scrum,skill__sql,skill__statistics,skill_agile,skill_aws,skill_budgeting,skill_c++,skill_cold_calling,skill_communication,skill_conflict_resolution,skill_crm,skill_deep_learning,skill_git,skill_hr_policies,skill_java,skill_leadership,skill_machine_learning,skill_negotiation,skill_networking,skill_presentation,skill_python,skill_r,skill_recruitment,skill_scheduling,skill_scrum,skill_sql,skill_statistics
Gender,1.000000,0.013598,-0.024370,-0.011902,-0.013479,-0.000863,-0.031015,-0.139215,-0.124238,0.000231,0.016513,0.009984,0.011234,-0.026204,0.031664,-0.013768,0.014226,-0.019949,-0.020306,-0.004868,-0.006229,-0.002439,0.005201,-0.032213,0.009831,0.015385,-0.010206,-0.028774,0.042564,0.005183,-0.010180,0.025866,-0.018548,0.000258,-0.028955,-0.024071,0.011705,0.010639,0.003662,0.027914,-0.035724,0.031386,0.016192,0.000424,0.030868,0.020216,-0.001442,0.039423,-0.023934,-0.005134,-0.028763,-0.007536,0.011705,-0.012405,0.003916,-0.010258,-0.029479
Race,0.013598,1.000000,-0.026104,-0.008531,-0.013117,-0.021636,-0.035884,-0.031625,-0.041165,-0.007720,0.001057,-0.001815,-0.013844,-0.000655,-0.024365,-0.003348,0.026565,-0.001254,0.022245,0.033167,0.017334,-0.027445,-0.019795,-0.016223,0.030746,0.018954,-0.029128,-0.008598,-0.004027,0.023544,-0.022217,0.018846,0.002839,-0.003936,0.002465,-0.014806,0.001553,0.005923,0.021717,-0.013429,-0.008186,-0.013348,0.007536,-0.008186,-0.001777,0.010067,0.007432,0.038899,0.011121,-0.006094,0.014926,0.010504,-0.008114,-0.015502,-0.001617,-0.024264,-0.011196
Age_Group,-0.024370,-0.026104,1.000000,0.000078,-0.036980,-0.006754,0.837926,0.008416,0.019073,-0.024058,0.006676,0.013538,0.004387,-0.004650,-0.017758,-0.012639,0.014614,-0.023116,0.003657,0.002458,0.005221,0.010511,-0.020605,0.042651,0.018205,-0.000052,0.007343,0.003391,0.009808,-0.004184,-0.013796,-0.008558,-0.018936,0.028016,0.002638,-0.017256,-0.000445,-0.000901,0.008951,0.009603,0.018152,0.001416,0.008286,-0.020147,0.009947,-0.013142,-0.008026,-0.004985,-0.007963,0.044252,-0.019863,0.003049,-0.005867,-0.005614,-0.003566,0.015641,-0.041729
Education,-0.011902,-0.008531,0.000078,1.000000,-0.003336,0.010159,-0.006599,0.014228,0.045803,-0.030620,0.025195,-0.028369,0.015981,0.023302,0.049862,0.007477,-0.005357,-0.003448,-0.018771,0.004140,0.001065,0.000685,-0.017071,-0.024114,-0.007824,-0.004538,0.014420,-0.032034,0.015590,0.001414,0.006645,0.007562,-0.010158,-0.011439,-0.013826,0.003816,0.013218,-0.018902,-0.010341,0.030118,0.016915,-0.001636,0.034572,0.021952,-0.019823,-0.002361,0.004986,0.002049,0.001341,-0.012615,0.032691,-0.007050,-0.010077,-0.001131,-0.002538,-0.055142,0.014680
Job_Role_Applied,-0.013479,-0.013117,-0.036980,-0.003336,1.000000,-0.001089,-0.031896,-0.015013,-0.019207,0.001190,-0.396158,0.001112,-0.394244,-0.197992,0.196861,0.212038,-0.209191,0.386756,-0.337527,0.198802,-0.388460,0.150534,0.364703,-0.203644,-0.197516,-0.195124,-0.007818,0.401286,0.203121,0.001210,0.001134,-0.032693,0.387737,0.000798,-0.257631,0.000843,-0.269505,-0.145590,0.146638,0.148465,-0.133074,0.280022,-0.287137,0.134591,-0.250825,0.098389,0.261789,-0.136294,-0.135655,-0.144379,-0.008261,0.246655,0.135901,0.000802,0.000795,0.033305,0.253632
Certifications,-0.000863,-0.021636,-0.006754,0.010159,-0.001089,1.000000,-0.011539,0.015105,-0.002280,0.003857,-0.003853,0.030167,-0.025541,-0.006704,0.014396,0.040290,-0.002237,-0.027430,0.007317,0.006308,-0.024513,0.028901,-0.034056,-0.007422,-0.028523,0.004349,0.014399,-0.009745,-0.001855,-0.005327,0.010396,0.006181,-0.011366,-0.006514,-0.007668,0.026483,0.03